# Schema drift demo

In [0]:
%run ./00_setup_and_logging

# Setup and logging

Using workspace.dav_project


## Bronze tables

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


ok: bronze_quarantine


...

ok: silver_commit, silver_issue, silver_pull_request, silver_quarantine


## Log table

ok: pipeline_execution_logs


## Small helper functions

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics


In [0]:
dbutils.widgets.text("demo_batch_id", "schema_drift_demo", "Batch ID for this demo (leave as is)")
DEMO_BATCH_ID = dbutils.widgets.get("demo_batch_id").strip() or "schema_drift_demo"
RUN_ID = new_run_id()
print("demo batch_id:", DEMO_BATCH_ID, "| run_id:", RUN_ID)

demo batch_id: schema_drift_demo | run_id: 9eae614f-469d-498d-804d-df5135d2e0e0


## Step 1: making 4 fake commit records

In [0]:
import json
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType,
)

BRONZE_SCHEMA = StructType([
    StructField("record_key", StringType(), False),
    StructField("source_repo", StringType(), False),
    StructField("entity_type", StringType(), False),
    StructField("raw_json", StringType(), False),
    StructField("api_params", StringType(), True),
    StructField("load_type", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("load_timestamp", TimestampType(), False),
])


def commit_json(sha, extra_field=False, bad_parents=False):
    rec = {
        "sha": sha,
        "commit": {
            "message": "Demo commit for the schema drift test",
            "author": {"name": "Demo Author", "email": "demo@users.noreply.github.com",
                       "date": "2026-01-01T10:00:00Z"},
            "committer": {"name": "Demo Author", "email": "demo@users.noreply.github.com",
                         "date": "2026-01-01T10:00:00Z"},
        },
        "author": {"login": "demo-author"},
        "committer": {"login": "demo-author"},
        "parents": "oops-not-a-list" if bad_parents else [{"sha": "parent-of-" + sha}],
    }
    if extra_field:
        rec["stats"] = {"additions": 10, "deletions": 2, "total": 12}
    return json.dumps(rec)


now = now_utc()
demo_rows = [
    ("demo-A-control",    "demo/fake-repo", "commits", commit_json("demo-A-control"),
     None, "demo", DEMO_BATCH_ID, RUN_ID, now),
    ("demo-B-newfield",   "demo/fake-repo", "commits", commit_json("demo-B-newfield", extra_field=True),
     None, "demo", DEMO_BATCH_ID, RUN_ID, now),
    ("demo-C-wrongshape", "demo/fake-repo", "commits", commit_json("demo-C-wrongshape", bad_parents=True),
     None, "demo", DEMO_BATCH_ID, RUN_ID, now),
    ("demo-D-brokenjson", "demo/fake-repo", "commits", '{"sha": "demo-D-brokenjson", "commit": {oops not json',
     None, "demo", DEMO_BATCH_ID, RUN_ID, now),
]

demo_df = spark.createDataFrame(demo_rows, schema=BRONZE_SCHEMA)
demo_df.createOrReplaceTempView("drift_demo_batch")
spark.sql(f"""
    MERGE INTO {CATALOG}.{SCHEMA}.bronze_commits AS t
    USING drift_demo_batch AS s
    ON t.record_key = s.record_key
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
print("4 fake records written to bronze_commits, batch_id =", DEMO_BATCH_ID)

4 fake records written to bronze_commits, batch_id = schema_drift_demo


## Step 2: running through Silver logic

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType

PII_SALT = "dav-project-2026-fixed-salt-change-if-you-want-but-keep-it-the-same"

COMMIT_JSON_SCHEMA = StructType([
    StructField("sha", StringType()),
    StructField("commit", StructType([
        StructField("message", StringType()),
        StructField("author", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
            StructField("date", StringType()),
        ])),
        StructField("committer", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
            StructField("date", StringType()),
        ])),
    ])),
    StructField("author", StructType([StructField("login", StringType())])),
    StructField("committer", StructType([StructField("login", StringType())])),
    StructField("parents", ArrayType(StructType([StructField("sha", StringType())]))),
])
TS_FMT = "yyyy-MM-dd'T'HH:mm:ss'Z'"


def hash_key(*cols):
    picked = F.coalesce(*cols, F.lit("unknown"))
    return F.sha2(F.concat(F.lit(PII_SALT), picked), 256)


bronze_demo = spark.table(f"{CATALOG}.{SCHEMA}.bronze_commits").filter(F.col("batch_id") == DEMO_BATCH_ID)
parsed = bronze_demo.withColumn("data", F.from_json(F.col("raw_json"), COMMIT_JSON_SCHEMA))

bad = parsed.filter(F.col("data").isNull() | F.col("data.sha").isNull())
good = parsed.filter(F.col("data").isNotNull() & F.col("data.sha").isNotNull())

silver_batch = good.select(
    F.col("data.sha").alias("sha"),
    F.col("source_repo").alias("repo"),
    hash_key(F.col("data.author.login"), F.col("data.commit.author.email")).alias("author_key"),
    hash_key(F.col("data.committer.login"), F.col("data.commit.committer.email")).alias("committer_key"),
    F.to_timestamp(F.col("data.commit.author.date"), TS_FMT).alias("committed_at"),
    F.length(F.col("data.commit.message")).alias("message_length"),
    (F.size(F.coalesce(F.col("data.parents"), F.array())) > 1).alias("is_merge_commit"),
    F.lit(now_utc()).alias("load_timestamp"),
).dropDuplicates(["sha"])

inserted = updated = 0
if silver_batch.limit(1).count() > 0:
    silver_batch.createOrReplaceTempView("drift_silver_batch")
    full_table = f"{CATALOG}.{SCHEMA}.silver_commit"
    spark.sql(f"""
        MERGE INTO {full_table} AS t
        USING drift_silver_batch AS s
        ON t.sha = s.sha
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    m = get_merge_metrics(full_table)
    inserted, updated = m["inserted"], m["updated"]

bad_count = bad.count()
if bad_count > 0:
    bad.select(
        F.lit("commit").alias("entity_type"),
        F.coalesce(F.col("record_key"), F.lit("unknown")).alias("record_key"),
        F.col("source_repo").alias("repo"),
        F.lit("could not parse commit JSON or missing sha").alias("reason"),
        F.col("raw_json"),
        F.col("batch_id"),
        F.lit(RUN_ID).alias("run_id"),
        F.lit(now_utc()).alias("load_timestamp"),
    ).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quarantine")

log_run(RUN_ID, "Schema-Drift-Demo", "commits", {"batch_id": DEMO_BATCH_ID},
        now_utc(), now_utc(), "SUCCESS", inserted, updated, bad_count)

print(f"good rows (reached silver_commit): {good.count()}")
print(f"bad rows (quarantined): {bad_count}")
print("Nothing crashed. That's the whole point.")

[log] Schema-Drift-Demo / commits -> SUCCESS (inserted=0, updated=3, quarantined=1)
good rows (reached silver_commit): 3
bad rows (quarantined): 1
Nothing crashed. That's the whole point.


## Step 3: displaying

In [0]:
display(spark.sql(f"""
    SELECT sha, message_length, is_merge_commit
    FROM {CATALOG}.{SCHEMA}.silver_commit
    WHERE sha LIKE 'demo-%'
    ORDER BY sha
"""))

display(spark.sql(f"""
    SELECT record_key, reason
    FROM {CATALOG}.{SCHEMA}.silver_quarantine
    WHERE record_key LIKE 'demo-%'
    ORDER BY record_key
"""))

sha,message_length,is_merge_commit
demo-A-control,37,false
demo-B-newfield,37,false
demo-C-wrongshape,37,false


record_key,reason
demo-D-brokenjson,could not parse commit JSON or missing sha
demo-D-brokenjson,could not parse commit JSON or missing sha
demo-D-brokenjson,could not parse commit JSON or missing sha


## Step 4: adopting new field, using mergeSchema

In [0]:
STATS_SCHEMA = StructType([
    StructField("sha", StringType()),
    StructField("stats", StructType([StructField("additions", LongType())])),
])

stats_batch = (
    bronze_demo
    .withColumn("d", F.from_json(F.col("raw_json"), STATS_SCHEMA))
    .select(
        F.col("d.sha").alias("sha"),
        F.col("d.stats").isNotNull().alias("has_extra_stats"),
    )
    .filter(F.col("sha").isNotNull())
)

print("Before adding the column:")
display(spark.sql(f"DESCRIBE TABLE {CATALOG}.{SCHEMA}.silver_commit"))

stats_batch.createOrReplaceTempView("drift_stats_batch")
spark.sql(f"""
    MERGE WITH SCHEMA EVOLUTION INTO {CATALOG}.{SCHEMA}.silver_commit AS t
    USING drift_stats_batch AS s
    ON t.sha = s.sha
    WHEN MATCHED THEN UPDATE SET *
""")

print("\nAfter adding the column:")
display(spark.sql(f"DESCRIBE TABLE {CATALOG}.{SCHEMA}.silver_commit"))

display(spark.sql(f"""
    SELECT sha, has_extra_stats
    FROM {CATALOG}.{SCHEMA}.silver_commit
    WHERE sha LIKE 'demo-%'
    ORDER BY sha
"""))

Before adding the column:


col_name,data_type,comment
sha,string,null
repo,string,null
author_key,string,null
committer_key,string,null
committed_at,timestamp,null
message_length,int,null
is_merge_commit,boolean,null
load_timestamp,timestamp,null



After adding the column:


col_name,data_type,comment
sha,string,null
repo,string,null
author_key,string,null
committer_key,string,null
committed_at,timestamp,null
message_length,int,null
is_merge_commit,boolean,null
load_timestamp,timestamp,null
has_extra_stats,boolean,null


sha,has_extra_stats
demo-A-control,false
demo-B-newfield,true
demo-C-wrongshape,false


## results
- A new, unexpected field never breaks anything. It is just not there.
- A field in the wrong shape never breaks the row. Only that one field
  goes blank.
- A truly broken record goes to quarantine, with a reason, instead of
  stopping the whole pipeline.
- A new column can be added to a real table on purpose, any time,
  without losing or rebuilding anything.

In [0]:
display(spark.sql(f"""
    SELECT run_id, layer, entity, status, rows_inserted, rows_updated, rows_quarantined, start_time
    FROM {CATALOG}.{SCHEMA}.pipeline_execution_logs
    WHERE run_id = '{RUN_ID}'
    ORDER BY start_time
"""))

run_id,layer,entity,status,rows_inserted,rows_updated,rows_quarantined,start_time
9eae614f-469d-498d-804d-df5135d2e0e0,Schema-Drift-Demo,commits,SUCCESS,0,3,1,2026-10-11T11:15:39.093Z
